HMC collect Server List

In [6]:
import requests
import xmltodict
import xml.etree.ElementTree as ET
from requests.packages.urllib3.exceptions import InsecureRequestWarning
import ssl

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
ssl_context = ssl.create_default_context(purpose=ssl.Purpose.CLIENT_AUTH)
ssl_context.verify_mode = ssl.CERT_NONE

user='hscroot'
passwd='Welc0me123'
HMCname='10.125.48.40'
#logonheaders = {'Content-Type': 'application/vnd.ibm.powervm.web+xml; type=LogonRequest'}
#logonUrl =  'https://'+HMCname+':12443/rest/api/web/Logon'
#logonPayload = '<LogonRequest schemaVersion="V1_0" xmlns="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/" xmlns:mc="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/"><UserID>' + user + '</UserID><Password>' + passwd + '</Password></LogonRequest>'
#ret = requests.put(logonUrl,data=logonPayload,headers=logonheaders,verify=False)
#if ret.status_code != 200:
#        print("Error: Logon failed error code=%d url=%s" %(ret.status_code,logonUrl))
#        if self.debug:
#                print("DEBUG: Returned:%s" %(ret.text))
#        # Do not return if we failed to logon
#        sys.exit(ret.status_code)
#xmlResponse = ET.fromstring(ret.text)
#token = xmlResponse[1].text
#print(token)
##print(ret.text)

In [7]:
def loginHmc(user,passwd,HMCname):
    logonheaders = {'Content-Type': 'application/vnd.ibm.powervm.web+xml; type=LogonRequest'}
    logonUrl =  'https://'+HMCname+':12443/rest/api/web/Logon'
    logonPayload = '<LogonRequest schemaVersion="V1_0" xmlns="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/" xmlns:mc="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/"><UserID>' + user + '</UserID><Password>' + passwd + '</Password></LogonRequest>'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.put(logonUrl,data=logonPayload,headers=logonheaders,proxies=proxies,verify=False)

    if ret.status_code != 200:
        print("Error: Logon failed error code=%d url=%s" %(ret.status_code,logonUrl))
        if self.debug:
                print("DEBUG: Returned:%s" %(ret.text))
        # Do not return if we failed to logon
        sys.exit(ret.status_code)
    xmlResponse = ET.fromstring(ret.text)
    token = xmlResponse[1].text
    return token

In [3]:
def logoffHmc(hmc,token):
    logoffheaders = {'X-API-Session' : token }
    logoffUrl =  'https://'+hmc+':12443/rest/api/web/Logon'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.delete(logoffUrl,headers=logoffheaders,proxies=proxies,verify=False)
    rcode = ret.status_code
    if rcode == 200 or rcode == 202 or rcode == 204:
        print("DEBUG:Successfully disconnected from the HMC (code=%d)" %(rcode))
    else:
        print("Error: Logoff failed error code=%d url=%s data=%s" %(rcode, logoffUrl, ret.text))
        sys.exit(rcode)

In [8]:
#logoffHmc('10.1.80.26',token)
logoffHmc(HMCname,token)

NameError: name 'token' is not defined

In [9]:
c

NameError: name 'c' is not defined

In [10]:
# HMC ye bağlanıp LogicalPArtition bilgilerini çekiyorsun.

prefHeaders = {'X-API-Session' : token}
prefUrl = 'https://'+HMCname+':12443/rest/api/uom/LogicalPartition'
proxies = {
                "http": "",
                "https": "",
                }
ret = requests.get(prefUrl, headers=prefHeaders,proxies=proxies,verify=False).content#.decode('utf-8')
#print(ret)
out=xmltodict.parse(ret)
#PartitionName 
#OperatingSystemVersion
#LogicalSerialNumber 
#CurrentProcessorCompatibilityMode
#ResourceMonitoringIPAddress
#SystemName

NameError: name 'token' is not defined

In [1]:
#for lpars in range(0,len(out.get('feed').get('entry'))):
#    print(str(lpars)+":"+out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('PartitionName').get('#text'))
#    #print(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('VirtualFibreChannelClientAdapters'))

In [11]:
#print(out.get('feed').get('entry')[2].get('content').get('LogicalPartition:LogicalPartition').get('SystemName').get('#text'))
#print(out.get('feed').get('entry')[2]).get('content')

In [12]:
import pandas as pd
SystemName=[]
PartitionID=[]
PartitionName=[]
PartitionIPAddress=[]
PartitionOS=[]
PartitionType=[]
PartitionState=[]
PartitionUUID=[]
PartitionMode=[]

for lpars in range(0,len(out.get('feed').get('entry'))):
    #print(lpars)
    SystemName.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('SystemName').get('#text'))
    PartitionID.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('PartitionID').get('#text'))
    PartitionName.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('PartitionName').get('#text'))

    #PartitionIPAddress.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('ResourceMonitoringIPAddress').get('#text')) 
    
    PartitionOS.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemType').get('#text'))
    PartitionType.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemVersion').get('#text'))
    PartitionState.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('PartitionState').get('#text'))
    PartitionUUID.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('PartitionUUID').get('#text'))
    PartitionMode.append(out.get('feed').get('entry')[lpars].get('content').get('LogicalPartition:LogicalPartition').get('CurrentProcessorCompatibilityMode').get('#text'))

df = pd.DataFrame(
{"SystemName":SystemName
,"PartitionID":PartitionID
,"PartitionName":PartitionName
#,"PartitionIPAddress":PartitionIPAddress
,"PartitionOS":PartitionOS
,"PartitionType":PartitionType
,"PartitionState":PartitionState
,"PartitionUUID":PartitionUUID
,"PartitionMode":PartitionMode}
)
#df["DateVal"]= pd.Timestamp.today().strftime('%Y-%m-%d %H:%M:%S')

df

,SystemName,PartitionID,PartitionName,PartitionOS,PartitionType,PartitionState,PartitionUUID,PartitionMode
0,DCMIPPHPWR002-SN7837B88,25,dcmipdbhan271,Linux,Linux/SLES 5.3.18-150300.59.153-defa15-SP3 15-SP3,running,369BF84C-8DBC-4741-8D30-C58EA4C896C4,POWER8
1,DCMIPPHPWR007-SN027BDA8,18,dcmipvmprd014,AIX,AIX 7.2 7200-05-07-2346,running,42F284F3-E39D-463E-9F3A-955E337C165A,POWER9_Base
2,DCMIPPHPWR002-SN7837B88,26,dcmipvmpx3001,Linux,Linux/SLES 5.3.18-150300.59.153-defa15-SP3 15-SP3,running,13BE4F6C-1A48-4B22-BDAD-29B730ECF5FC,POWER8
3,DCMIPPHPWR004-SN025D068,9,dcmievmpe1004,Linux,Linux/SLES 5.3.18-150300.59.133-defa15-SP3 15-SP3,running,3E0C3033-665D-4C88-ACFA-6F53A6C3609D,POWER9_Base
4,DCMIPPHPWR002-SN7837B88,29,dcmievmgpn003,AIX,AIX 7.2 7200-05-07-2346,running,0A4C7840-7124-4260-BF09-09B01F6CE202,POWER8
...,...,...,...,...,...,...,...,...
128,DCMIPPHPWR001-SN7837BC8,28,dcmipdbhan272,Linux,Linux/SLES 5.3.18-150300.59.153-defa15-SP3 15-SP3,running,3180A90F-8209-4D67-B164-1503228BDBD3,POWER8
129,DCMIPPHPWR003-SN025D0A8,4,dcmipvmpe1011,Linux,Linux/SLES 5.3.18-150300.59.141-defa15-SP3 15-SP3,running,770C850B-6DF4-4BFF-817A-F839E8744964,POWER9_Base
130,DCMIPPHPWR006-SN027BDB8,37,dcmipvmgml010,AIX,AIX 7.2 7200-05-07-2346,running,69B2C6DE-4882-4FAC-949C-4906250192E4,POWER9_Base
131,DCMIPPHPWR002-SN7837B88,17,d1mipvmchi001,AIX,AIX 7.2 7200-05-07-2346,running,35A6D31A-1D0E-4539-9645-8451C58A3098,POWER8


In [9]:
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('SystemName').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionID').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionName').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('ResourceMonitoringIPAddress').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemType').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemVersion').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionState').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionUUID').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('CurrentProcessorCompatibilityMode').get('#text'))


DCMIPPHPWR002-SN7837B88
16
dcmipvmpe3004
130.172.132.189
Linux
Linux/SLES 5.3.18-150300.59.153-defa15-SP3 15-SP3
running
7D0475AD-E823-4F9B-8D8C-9DE31CD1ED5D
POWER8


In [10]:
out.get('feed').get('entry')[6].get('content').get('LogicalPartition:LogicalPartition').get('ResourceMonitoringIPAddress').get('#text')

AttributeError: 'NoneType' object has no attribute 'get'

In [11]:
try: out.get('feed').get('entry')[6].get('content').get('LogicalPartition:LogicalPartition').get('ResourceMonitoringIPAddress').get('#text') 
except AttributeError: 
            print("0")

0
